In [1]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

In [2]:
import tensorflow as tf
from typing import Any

2026-01-21 23:40:47.053853: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-21 23:40:47.161977: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-21 23:40:47.193679: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-21 23:40:47.451682: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-21 23:40:49.347584: W tensorflow/compiler/tf2

In [3]:
from mobilenetv2ssd.models.ssd.ops.heads_tf import *
from mobilenetv2ssd.models.mobilenet_v2.backbone import build_backbone
from mobilenetv2ssd.models.ssd.fpn import ExtraFeaturePyramid

In [4]:
from mobilenetv2ssd.core.config import load_config

In [26]:
from mobilenetv2ssd.models.ssd.orchestration.priors_orch import build_priors_from_config

In [24]:
main_cfg_path = "configs/train/default.yaml"
model_cfg_path = "configs/model/mobilenetv2_ssd_voc.yaml"
data_cfg_path = "configs/data/voc_224.yaml"
eval_cfg_path = "configs/eval/default.yaml"

In [28]:
config = load_config(main_cfg_path,model_cfg_path,data_cfg_path,eval_cfg_path)

In [30]:
priors, priors_meta = build_priors_from_config(config)

In [32]:
priors

<tf.Tensor: shape=(13502, 4), dtype=float32, numpy=
array([[0.01315789, 0.01315789, 0.2       , 0.2       ],
       [0.01315789, 0.01315789, 0.26457512, 0.26457512],
       [0.01315789, 0.01315789, 0.28284273, 0.14142136],
       ...,
       [0.75      , 0.75      , 1.        , 0.6892024 ],
       [0.75      , 0.75      , 0.67175144, 1.        ],
       [0.75      , 0.75      , 0.6892024 , 1.        ]], dtype=float32)>

In [33]:
from datasets.voc import build_voc_dataset
from datasets.transforms import build_train_transforms
from datasets.collate import create_training_dataset

In [34]:
cls_head = {'norm_cfg': 'batch_norm',
  'head_type': 'conv3x3',
  'intermediate_channels': 256,
  'squeeze_ratio': 1.0,
  'use_sigmoid_cls': False}

extra_levels = [{'name': 'P6', 'out_channels': 256, 'stride': 2, 'kernel_size': 3},
  {'name': 'P7', 'out_channels': 256, 'stride': 2, 'kernel_size': 3},
  {'name': 'P8', 'out_channels': 128, 'stride': 2, 'kernel_size': 3}],

base = 'C5'

In [35]:
compose = build_train_transforms(config)

In [36]:
data = build_voc_dataset(config, split = "train", transform = compose)

In [37]:
class SSD(tf.keras.Model):
    def __init__(self, backbone_type: str, name: str, feature_maps: list[str], number_of_classes: int, number_of_anchors_per_layer: list[int], input_shape: tuple[int,int,int], loc_head_configuration: dict[str, Any],cls_head_configuration: dict[str, Any],extra_levels: list[dict[str,Any]], extra_base: str | None, alpha: float = 1.0 ,**kwargs):
        super().__init__(name=name, **kwargs)
        
        self.feature_maps = feature_maps
        self.backbone = build_backbone(input_shape = input_shape,alpha = alpha, name = backbone_type)
        
        # Initializing the config for the extra heads
        self.extra_pyramid = ExtraFeaturePyramid(name = "extra_pyramid", extra_levels_cfg = extra_levels)
        self.extra_base = feature_maps[-1] if extra_base is None else extra_base

        # Localization and Classification Heads
        self.localization_head = LocalizationHead(name = "loc_head", num_anchors_per_location = number_of_anchors_per_layer, head_type = loc_head_configuration.get("head_type","conv3x3"), initial_norm_strategy = loc_head_configuration.get("initial_norm_strategy","BatchNorm"), squeeze_ratio = loc_head_configuration.get("squeeze_ratio",1.0), intermediate_conv = loc_head_configuration.get("intermediate_conv",128), in_channels =  loc_head_configuration.get("in_channels",[256,512,512])) 
        self.classification_head = ClassificationHead(name = "loc_head", num_anchors_per_location = number_of_anchors_per_layer, number_of_classes = number_of_classes , head_type = cls_head_configuration.get("head_type","conv3x3"), norm_cfg = cls_head_configuration.get("initial_norm_strategy","BatchNorm"), squeeze_ratio = cls_head_configuration.get("squeeze_ratio",1.0), intermediate_conv = cls_head_configuration.get("intermediate_conv",128))
        
    
    def call(self,image: tf.Tensor,training = False):
        
        feature_map_dict = self.backbone(image, training = training)

        feature_maps = [feature_map_dict[key] for key in self.feature_maps]

        # Getting the base feature for the pyramid
        base = feature_map_dict[self.extra_base]

        # Passing it through the extra pyramid
        extra_features = self.extra_pyramid(base,training = training)
        all_features = feature_maps + extra_features

        # Pass through the localization and classification heads
        pred_offsets = self.localization_head(all_features, training = training)
        pred_logits = self.classification_head(all_features, training = training)

        return pred_offsets, pred_logits

    def build(self,input_shape):

        # Building the model
        dummy_image = tf.zeros((1,) + tuple(input_shape[1:])) 
        feature_dict = self.backbone(dummy_image,training = False)

        # Getting the base feature
        feature_maps = [feature_dict[key] for key in self.feature_maps]
        base = feature_dict[self.extra_base]

        # Passing it through the Extra Pyramid
        extra_features = self.extra_pyramid(base,training = False)
        all_features = feature_maps + extra_features

        # Pass through the localization and classification heads
        _ = self.localization_head(all_features, training = False)
        _ = self.classification_head(all_features, training = False)

        super().build(input_shape)

In [38]:
ssd = SSD(backbone_type = "mobilenetv2", 
    name = "mobilenetv2-ssd", 
    number_of_classes = 20,
    number_of_anchors_per_layer = [6, 10, 10, 6, 6, 6],
    feature_maps = ["C3", "C4", "C5"],
    input_shape = [300, 300, 3],
    loc_head_configuration = {'norm_cfg':"batch_norm",'head_type':"conv3x3",'intermediate_channels':256, 'squeeze_ratio':1.0},
    cls_head_configuration = {'norm_cfg':"batch_norm",'head_type':"conv3x3",'intermediate_channels':256, 'squeeze_ratio':1.0, 'use_sigmoid_cls': False},
    extra_levels = extra_levels,
    extra_base = base)

[build_backbone] Loading existing weights from: /mnt/d/dev/MobileNetV2-SSD/src/mobilenetv2ssd/models/mobilenet_v2/weights/mobilenetv2_imagenet_notop_300x300_1.weights.h5


In [39]:
pipeline = create_training_dataset(config, data)

In [40]:
batch = next(iter(pipeline))

In [42]:
ssd(batch['image'], training = False)

2026-01-22 00:21:36.533336: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8907
W0000 00:00:1769059296.631661     741 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769059296.676967     741 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769059296.690079     741 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769059296.699670     741 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769059296.717715     741 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769059296.732807     741 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769059296.736780     741 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769059296.896896     741 gpu_t

TypeError: list indices must be integers or slices, not str